In [1]:
# Check python version
!python --version

Python 3.9.12


In [2]:
# import libraries
import pandas as pd
import pickle

from sklearn.feature_extraction import DictVectorizer

from sklearn.metrics import root_mean_squared_error

In [3]:
import mlflow

mlflow.set_tracking_uri("sqlite:///backend.db")
mlflow.set_experiment("green_tripdata_2026-01-experiment")

2026/06/12 06:45:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/12 06:45:51 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp-megh/03-orchestration/mlruns/1', creation_time=1781245547721, experiment_id='1', last_update_time=1781245547721, lifecycle_stage='active', name='green_tripdata_2026-01-experiment', tags={}>

In [4]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    ## We dont need this anymore as the parquet file already has the correct datatypes. We can remove this code to speed up the processing.
    # df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    # df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime

    df.duration = df.duration.apply(lambda x: x.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']

    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [5]:
df_train = read_dataframe("../data/green_tripdata_2026-01.parquet")
df_val = read_dataframe("../data/green_tripdata_2026-02.parquet")

In [ ]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [7]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [12]:
import xgboost as xgb
from mlflow.models.signature import infer_signature
from pathlib import Path

In [13]:
Path("models/").mkdir(exist_ok=True)

In [9]:
with mlflow.start_run():
    
    mlflow.set_tag("Developer Name", "Megh Modi")
    mlflow.log_param("model_type", "XGBoost Regression")
    
    params = {
        "learning_rate": 0.0784901317383746,
        "max_depth": 84,
        "min_child_weight": 0.49036887395182704,
        "objective": "reg:linear",
        "reg_alpha": 0.014718634417135872,
        "reg_lambda": 0.0038376926340167993,
        "seed": 42
    }
    
    mlflow.log_params(params)
    
    booster = xgb.train(
        params=params,
        dtrain=xgb.DMatrix(X_train, label=y_train),
        num_boost_round=100,
        evals=[(xgb.DMatrix(X_val, label=y_val), 'validation')],
        early_stopping_rounds=50
    )
    
    y_pred = booster.predict(xgb.DMatrix(X_val, label=y_val))
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    with open('models/preprocessor.b', 'wb') as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact(local_path="models/preprocessor.b", artifact_path="preprocessor")

    # Create an input example and signature so MLflow records the model signature
    input_example = X_val[:5].toarray() if hasattr(X_val[:5], "toarray") else X_val[:5]
    signature = infer_signature(input_example, y_val[:5])

    mlflow.xgboost.log_model(booster, artifact_path="models", signature=signature, input_example=input_example)

/home/codespace/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:45:56] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:9.86997
[1]	validation-rmse:9.41984
[2]	validation-rmse:9.02020
[3]	validation-rmse:8.66048
[4]	validation-rmse:8.34140
[5]	validation-rmse:8.06935
[6]	validation-rmse:7.81475
[7]	validation-rmse:7.61568
[8]	validation-rmse:7.41459
[9]	validation-rmse:7.26222
[10]	validation-rmse:7.11302
[11]	validation-rmse:7.00106
[12]	validation-rmse:6.88760
[13]	validation-rmse:6.80071
[14]	validation-rmse:6.71993
[15]	validation-rmse:6.64951
[16]	validation-rmse:6.58884
[17]	validation-rmse:6.53882
[18]	validation-rmse:6.49668
[19]	validation-rmse:6.46194
[20]	validation-rmse:6.43383
[21]	validation-rmse:6.40677
[22]	validation-rmse:6.38682
[23]	validation-rmse:6.36549
[24]	validation-rmse:6.35209
[25]	validation-rmse:6.33529
[26]	validation-rmse:6.32481
[27]	validation-rmse:6.31364
[28]	validation-rmse:6.30527
[29]	validation-rmse:6.29722
[30]	validation-rmse:6.29170
[31]	validation-rmse:6.28674
[32]	validation-rmse:6.28221
[33]	validation-rmse:6.27919
[34]	validation-rmse:6.2

2026/06/12 06:47:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/home/codespace/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:47:52] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
